# Multimodal AI Fundamentals + Agent Workflow Consolidation
### Practice Notebook (Deliverable Work Session)

**Assumed pre-installed libraries:** none required for Parts 1-2 (pure
Python). If you want to try a real multimodal API call in Part 3, you'll
need the `anthropic` package and an API key -- clearly marked and skippable.


## 1. Multimodal models: the conceptual bridge from Week 1

Recall from Week 1: text is tokenized into sub-word units, then each token
is mapped to a dense embedding vector, and the Transformer's attention
mechanism lets every token attend to every other token.

A vision-language model extends this same idea: an image is cut into
patches, and each patch is converted into a vector "token" that lives in
(roughly) the same embedding space as text tokens. This lets the *same*
attention mechanism attend across text tokens and image-patch tokens
together -- no fundamentally new architecture is required, just a new kind
of token.


In [1]:
def tokenize_text(text: str) -> list:
    """Simplified stand-in for text tokenization (Week 1, Day 4)."""
    return text.split()

def tokenize_image_patches(image_width: int, image_height: int, patch_size: int = 16) -> list:
    """Simplified stand-in for how a vision-language model turns an image
    into a sequence of patch 'tokens'. Returns a list of (row, col) patch
    coordinates -- in a real model each of these becomes an embedding vector,
    the same way each word becomes an embedding vector.
    """
    patches = []
    for row in range(0, image_height, patch_size):
        for col in range(0, image_width, patch_size):
            patches.append((row // patch_size, col // patch_size))
    return patches

text_tokens = tokenize_text("What is shown in this chart?")
image_tokens = tokenize_image_patches(image_width=224, image_height=224, patch_size=16)

print(f"Text tokens ({len(text_tokens)}): {text_tokens}")
print(f"Image patch tokens: {len(image_tokens)} total (a 224x224 image at 16x16 patches)")
print(f"First few image patch coordinates: {image_tokens[:5]}")
print()
print(f"Combined sequence length the attention mechanism sees: "
      f"{len(text_tokens) + len(image_tokens)} tokens (text + image)")


Text tokens (6): ['What', 'is', 'shown', 'in', 'this', 'chart?']
Image patch tokens: 196 total (a 224x224 image at 16x16 patches)
First few image patch coordinates: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]

Combined sequence length the attention mechanism sees: 202 tokens (text + image)


**Exercise 5.1:** Recompute `tokenize_image_patches` for a larger image
(e.g., 448x448) with the same `patch_size=16`. How does the number of image
tokens scale as image resolution increases? Given that attention cost grows
with the *square* of sequence length (from Week 1, Day 3), what practical
problem does this suggest for feeding very high-resolution images to a
vision-language model?


In [ ]:
image_tokens_large = tokenize_image_patches(
    image_width=448,
    image_height=448,
    patch_size=16
)

print(f"Image patch tokens: {len(image_tokens_large)}")

## 2. Common multimodal use cases (conceptual walkthrough)

Let's simulate the *shape* of a few common multimodal workflows -- not with
real image processing, but with stand-in functions that show what
information flows where.


In [2]:
def describe_image_stub(image_description_for_demo: str) -> str:
    """Stand-in for: 'a vision-language model looks at a real image and
    describes it.' We pass in a plain-text description here purely so the
    notebook works without real image files or an API key.
    # TODO: replace with a real multimodal LLM call, passing an actual image.
    """
    return f"(stub) Image shows: {image_description_for_demo}"

def extract_structured_data_from_receipt_stub(receipt_text_for_demo: str) -> dict:
    """Stand-in for: 'a vision-language model reads a scanned receipt image
    and extracts structured fields.'
    # TODO: replace with a real multimodal LLM call + a JSON output schema.
    """
    # Toy parsing logic standing in for real extraction.
    import re
    total_match = re.search(r"Total: Rs\.(\d+)", receipt_text_for_demo)
    return {
        "vendor": receipt_text_for_demo.split(",")[0],
        "total": total_match.group(1) if total_match else None,
    }

print(describe_image_stub("a bar chart showing quarterly revenue rising each quarter"))
print()
print(extract_structured_data_from_receipt_stub("Cafe Coffee Day, 2 items, Total: Rs.340"))


(stub) Image shows: a bar chart showing quarterly revenue rising each quarter

{'vendor': 'Cafe Coffee Day', 'total': '340'}


**Exercise 5.2:** For each use case below, briefly state (1-2 sentences)
what the "perceive" input would be, and what a plausible tool the agent
might call *afterward*, based on what it perceived:
1. A visually impaired user asks an app to describe what's in front of their
   phone camera.
2. An expense-reporting tool needs to log a photographed receipt.
3. A computer-use agent needs to click the "Submit" button on a webpage
   screenshot.


1. Visually impaired user: The perceive input would be the live image from the phone’s camera. After identifying objects, people, or obstacles, the agent could call a text-to-speech tool to describe the surroundings aloud.
2. Expense-reporting tool: The perceive input would be a photograph of the receipt. After extracting details such as the vendor, date, and total amount, the agent could call an expense-management tool to save the expense.
3. Computer-use agent: The perceive input would be a screenshot of the webpage showing the “Submit” button. After identifying the button’s position, the agent could call a mouse-click tool to click it.

## 3. (Optional) A real multimodal + tool-use call

If you have an `GROQ or HF API KEYS`, this is what a real multimodal request
looks like -- combining an image input with the tool-calling mechanics from
Day 2. This cell is optional and illustrative; read it even if you can't run
it.


In [ ]:
import base64, os

image_path = "images/"
with open(image_path, "rb") as f:
    image_b64 = base64.standard_b64encode(f.read()).decode("utf-8")
print(image_b64)
#
# client =  # Todo change client definition
#
# log_expense_tool = {
#     "name": "log_expense",
#     "description": "Log a business expense with a vendor name and total amount.",
#     "input_schema": {
#         "type": "object",
#         "properties": {
#             "vendor": {"type": "string"},
#             "total": {"type": "number"},
#         },
#         "required": ["vendor", "total"],
#     },
# }
# Todo: change the response code as per client used:
# response = client.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=500,
#     tools=[log_expense_tool],
#     messages=[{
#         "role": "user",
#         "content": [
#             {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_b64}},
#             {"type": "text", "text": "Log this receipt as an expense."},
#         ],
#     }],
# )
# for block in response.content:
#     if block.type == "tool_use":
#         print("Model wants to call:", block.name, "with input:", block.input)

print("This cell is illustrative -- uncomment, add an image file and API key to run it for real.")


## 4. DELIVERABLE: Simple agent workflow implementation

Now assemble a small end-to-end agent using the building blocks from the
whole week:

- Day 1: a perceive-reason-act loop
- Day 2: at least one tool with a proper schema
- Day 3: at least one workflow pattern (routing, orchestrator-worker, etc.)
- Day 4: at least one reasoning/decomposition or self-reflection step

Below is a starter scaffold implementing a **support-ticket triage agent**.
It's intentionally incomplete -- follow the `# TODO` markers to finish it, or
replace the whole scenario with one of your own choosing.


In [4]:
# --- Tools (Day 2) ---

def check_customer_account_tool(customer_id: str) -> dict:
    """FAKE tool. # TODO: replace with a real lookup / real LLM-driven call."""
    fake_accounts = {"C001": {"plan": "premium", "issues_this_month": 0},
                      "C002": {"plan": "free", "issues_this_month": 3}}
    return fake_accounts.get(customer_id, {"plan": "unknown", "issues_this_month": 0})

check_customer_account_schema = {
    "name": "check_customer_account_tool",
    "description": "Look up a customer's plan tier and how many issues they've filed this month.",
    "input_schema": {
        "type": "object",
        "properties": {"customer_id": {"type": "string"}},
        "required": ["customer_id"],
    },
}

# --- Routing step (Day 3) ---

def classify_ticket(ticket_text: str) -> str:
    """# TODO: replace with a real LLM call."""
    text = ticket_text.lower()
    if any(w in text for w in ["refund", "charge", "billing"]):
        return "billing"
    if any(w in text for w in ["bug", "crash", "error"]):
        return "technical"
    return "general"

# --- Reasoning / decision step (Day 4) ---

def decide_escalation(category: str, account_info: dict) -> bool:
    """# TODO: replace with a real LLM call reasoning over category + account_info.
    Simple rule for this scaffold: escalate technical issues for premium
    customers, or any customer with 3+ issues this month.
    """
    if account_info.get("plan") == "premium" and category == "technical":
        return True
    if account_info.get("issues_this_month", 0) >= 3:
        return True
    return False

# --- The agent loop (Day 1) ---

def triage_agent(ticket_text: str, customer_id: str) -> dict:
    # PERCEIVE
    print(f"[PERCEIVE] New ticket from {customer_id}: '{ticket_text}'")

    # REASON (classify)
    category = classify_ticket(ticket_text)
    print(f"[REASON] Classified as: {category}")

    # ACT (tool call)
    print("[ACT] Calling check_customer_account_tool...")
    account_info = check_customer_account_tool(customer_id)
    print(f"[PERCEIVE] Tool result: {account_info}")

    # REASON (decide escalation) -- TODO: add a self-reflection step here:
    # after deciding, have the agent double-check its own decision against
    # a stated policy (Day 4, self-correction) before finalizing.
    escalate = decide_escalation(category, account_info)
    print(f"[REASON] Escalate to human? {escalate}")

    # ACT (final)
    outcome = {
        "ticket": ticket_text,
        "category": category,
        "account_info": account_info,
        "escalated": escalate,
    }
    print(f"[ACT] Final routing decision: {outcome}")
    return outcome

triage_agent("The app keeps crashing when I try to check out.", "C001")


[PERCEIVE] New ticket from C001: 'The app keeps crashing when I try to check out.'
[REASON] Classified as: technical
[ACT] Calling check_customer_account_tool...
[PERCEIVE] Tool result: {'plan': 'premium', 'issues_this_month': 0}
[REASON] Escalate to human? True
[ACT] Final routing decision: {'ticket': 'The app keeps crashing when I try to check out.', 'category': 'technical', 'account_info': {'plan': 'premium', 'issues_this_month': 0}, 'escalated': True}


{'ticket': 'The app keeps crashing when I try to check out.',
 'category': 'technical',
 'account_info': {'plan': 'premium', 'issues_this_month': 0},
 'escalated': True}

**TODO for your deliverable:**
1. Add the missing self-reflection step noted in the code comment above (Day
   4): after `decide_escalation` runs, add a second check that reviews the
   decision against a stated policy (e.g., "never leave a billing issue for
   a paying customer unescalated") and can override the first decision.
2. Add a second tool of your own (Day 2-style: name, description, schema,
   and a real Python function).
3. Replace at least one `# TODO: replace with a real LLM call` with an
   actual Anthropic API call, following the pattern shown in Part 3 above.
4. Test your finished agent on at least 3 different example tickets and
   confirm the routing/escalation decisions make sense.


In [ ]:
```python
import os
import json
import anthropic

# Set ANTHROPIC_API_KEY in your environment before running.
client = anthropic.Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"]
)

# --- Tool 1: Customer account lookup ---

def check_customer_account_tool(customer_id: str) -> dict:
    """Look up a customer's plan and monthly issue count."""
    fake_accounts = {
        "C001": {"plan": "premium", "issues_this_month": 0},
        "C002": {"plan": "free", "issues_this_month": 3},
        "C003": {"plan": "free", "issues_this_month": 0},
    }
    return fake_accounts.get(
        customer_id,
        {"plan": "unknown", "issues_this_month": 0}
    )


check_customer_account_schema = {
    "name": "check_customer_account_tool",
    "description": "Look up a customer's plan tier and monthly issue count.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {"type": "string"}
        },
        "required": ["customer_id"],
    },
}


# --- Tool 2: Service status lookup ---

def check_service_status_tool(service_name: str) -> dict:
    """Check the locally recorded status of a service."""
    service_status = {
        "checkout": "degraded",
        "payments": "operational",
        "login": "operational",
    }
    status = service_status.get(
        service_name.lower(),
        "unknown"
    )
    return {
        "service": service_name,
        "status": status
    }


check_service_status_schema = {
    "name": "check_service_status_tool",
    "description": "Check whether a named service is operational or degraded.",
    "input_schema": {
        "type": "object",
        "properties": {
            "service_name": {"type": "string"}
        },
        "required": ["service_name"],
    },
}


# --- Routing step: Real Anthropic API call ---

def classify_ticket(ticket_text: str) -> str:
    """Use Claude to classify a support ticket."""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=50,
        messages=[
            {
                "role": "user",
                "content": (
                    "Classify this support ticket into exactly one "
                    "category: billing, technical, or general.\n"
                    "Billing includes refunds, charges and payments. "
                    "Technical includes crashes, bugs and errors. "
                    "Return only the category word.\n\n"
                    f"Ticket: {ticket_text}"
                ),
            }
        ],
    )

    category = response.content[0].text.strip().lower()

    if category not in {"billing", "technical", "general"}:
        raise ValueError(
            f"Unexpected category returned by Claude: {category}"
        )

    return category


# --- Reasoning / initial decision ---

def decide_escalation(category: str, account_info: dict) -> bool:
    """Apply the initial escalation rules."""
    if (
        account_info.get("plan") == "premium"
        and category == "technical"
    ):
        return True

    if account_info.get("issues_this_month", 0) >= 3:
        return True

    return False


# --- Self-reflection / policy verification ---

def reflect_on_decision(
    category: str,
    account_info: dict,
    service_info: dict,
    initial_decision: bool
) -> tuple[bool, str]:
    """Double-check the initial decision against explicit policies."""

    plan = account_info.get("plan")
    issues = account_info.get("issues_this_month", 0)
    status = service_info.get("status")

    # Policy 1: Premium customers with billing issues must be escalated.
    if plan == "premium" and category == "billing":
        return True, "Premium billing issue requires escalation."

    # Policy 2: Three or more issues in a month require escalation.
    if issues >= 3:
        return True, "Customer has at least three monthly issues."

    # Policy 3: Premium technical issues must be escalated.
    if plan == "premium" and category == "technical":
        return True, "Premium technical issue requires escalation."

    # Policy 4: A confirmed service degradation requires human review
    # for a related technical ticket.
    if category == "technical" and status == "degraded":
        return True, "Related service is degraded."

    # Retain the initial decision if no override is needed.
    return initial_decision, "Initial decision complies with policy."


# --- Agent loop ---

def triage_agent(ticket_text: str, customer_id: str) -> dict:
    print(f"\n[PERCEIVE] Ticket from {customer_id}: {ticket_text}")

    # Step 1: Classify using Claude.
    category = classify_ticket(ticket_text)
    print(f"[REASON] Category: {category}")

    # Step 2: Look up the customer account.
    account_info = check_customer_account_tool(customer_id)
    print(f"[ACT] Account lookup: {account_info}")

    # Step 3: Call the second tool for technical tickets.
    if category == "technical":
        text = ticket_text.lower()

        if "checkout" in text or "check out" in text:
            service_name = "checkout"
        elif "payment" in text or "pay " in text:
            service_name = "payments"
        elif "login" in text or "sign in" in text:
            service_name = "login"
        else:
            service_name = "unknown"

        service_info = check_service_status_tool(service_name)
    else:
        service_info = {
            "service": "not applicable",
            "status": "not checked"
        }

    print(f"[ACT] Service lookup: {service_info}")

    # Step 4: Make the initial escalation decision.
    initial_decision = decide_escalation(
        category, account_info
    )
    print(f"[REASON] Initial escalation: {initial_decision}")

    # Step 5: Self-reflect and apply policy overrides.
    final_decision, reflection_reason = reflect_on_decision(
        category,
        account_info,
        service_info,
        initial_decision
    )

    print(f"[REFLECTION] {reflection_reason}")
    print(f"[REFLECTION] Final escalation: {final_decision}")

    # Step 6: Return the final routing result.
    outcome = {
        "ticket": ticket_text,
        "customer_id": customer_id,
        "category": category,
        "account_info": account_info,
        "service_info": service_info,
        "initial_escalation": initial_decision,
        "escalated": final_decision,
        "reflection_reason": reflection_reason,
        "route": "human_support" if final_decision else "normal_queue",
    }

    print(f"[ACT] Final outcome: {json.dumps(outcome, indent=2)}")
    return outcome


# --- Test on three different tickets ---

test_cases = [
    (
        "The app keeps crashing when I try to check out.",
        "C001"
    ),
    (
        "I was charged twice and need a refund.",
        "C002"
    ),
    (
        "How can I change my profile display name?",
        "C003"
    ),
]

results = []

for ticket, customer_id in test_cases:
    results.append(triage_agent(ticket, customer_id))
```

## 5. DELIVERABLE: Agent workflow documentation

Once your agent is working, write up a short documentation section (as
markdown cells right here in the notebook, or as a separate `.md` file)
covering:

1. **Purpose** — what task does this agent perform, in one sentence?
2. **Tools** — list every tool the agent can call, with its name and a
   one-line description of what it does.
3. **Loop structure** — describe the perceive-reason-act steps in order,
   the way the `print()` statements in `triage_agent` trace them out.
4. **Workflow pattern(s) used** — which of Day 3's five patterns
   (chaining, routing, parallelization, orchestrator-worker,
   evaluator-optimizer) does your agent use, and where?
5. **Reasoning / self-reflection** — where does your agent reason about
   what to do next, and does it include any self-correction step? If not,
   identify one place where adding one would make the agent more robust.
6. **Known limitations** — what inputs or edge cases would break your
   agent, or cause it to make a bad decision? (Every real agent has some —
   documenting them honestly is part of the deliverable, not a sign of
   failure.)

This documentation habit is directly transferable to real-world agent
development, where undocumented agent behavior is a common source of
production incidents.


# Customer Support Ticket Triage Agent

## 1. Purpose
The agent classifies customer support tickets, checks customer account details and relevant service status, and decides whether a ticket should be escalated to a human support agent.

## 2. Tools

1. **`check_customer_account_tool`** — Retrieves the customer's subscription plan and the number of issues reported during the current month.
2. **`check_service_status_tool`** — Checks whether a service, such as checkout, payments, or login, is operational or experiencing problems.
3. **`classify_ticket`** — Uses the Anthropic Claude API to classify a ticket as billing, technical, or general.

The agent also uses `decide_escalation` to make an initial routing decision and `reflect_on_decision` to verify that decision against the escalation policies. These are Python functions rather than external tools.

## 3. Loop Structure: Perceive–Reason–Act

The agent follows these steps:

1. **Perceive:** Receives the customer's ticket text and customer ID.
2. **Reason:** Calls Claude to classify the ticket into billing, technical, or general.
3. **Act:** Calls `check_customer_account_tool` to retrieve the customer's account information.
4. **Act:** For technical tickets, calls `check_service_status_tool` to check the relevant service.
5. **Reason:** Calls `decide_escalation` to make an initial escalation decision based on the category and account information.
6. **Self-reflection:** Calls `reflect_on_decision` to check the decision against the escalation policies and override it when necessary.
7. **Act:** Produces the final result, including the category, account details, escalation decision, explanation, and destination queue.

The `print()` statements trace these steps and make the agent's behavior easier to debug.

## 4. Workflow Patterns Used

- **Chaining:** The agent performs classification, account lookup, initial decision-making, policy verification, and final routing in sequence.
- **Routing:** The ticket category determines the handling path. Technical tickets trigger an additional service-status lookup, while other categories skip that step.

The current implementation does not use parallelization, an orchestrator-worker pattern, or an evaluator-optimizer loop.

## 5. Reasoning and Self-Reflection

The initial reasoning occurs in `decide_escalation`, which checks whether a ticket should be escalated based on the customer's plan, issue count, and ticket category.

The self-correction step occurs in `reflect_on_decision`. It checks explicit policies, such as escalating billing issues from premium customers, escalating customers with three or more monthly issues, and escalating technical tickets involving a degraded service. This function can override the initial decision.

This improves reliability because the final decision is checked against stated policies instead of relying entirely on the initial rules.

## 6. Known Limitations

- **Mock account data:** Customer details are stored in a small hardcoded dictionary. Unknown customer IDs receive default values, which could lead to incorrect decisions.
- **Mock service status:** Service health information is predefined and may not reflect actual system conditions.
- **API dependency:** Ticket classification requires a valid Anthropic API key, network access, and an available model. API failures or unexpected responses can interrupt execution.
- **Classification errors:** Claude may misclassify ambiguous tickets, affecting subsequent decisions.
- **Limited categories:** Only billing, technical, and general categories are supported.
- **Simple service detection:** The service lookup uses keyword matching. It may fail to identify the correct service when customers describe an issue indirectly.
- **Incomplete escalation policies:** The policies cover only a few scenarios. Urgent security incidents, suspected fraud, or other high-risk issues may require additional rules.
- **No persistent storage:** Ticket decisions are returned and printed but are not saved to a database or ticket-management system.
- **No retry mechanism:** Temporary API failures are not automatically retried.
- **No real tool-calling protocol:** The tool schemas are defined for documentation, but the Python functions are called directly. Claude does not dynamically select and execute these tools.

## Conclusion

This project demonstrates how an AI-powered support agent can combine LLM-based classification, Python tools, conditional routing, rule-based decisions, and self-reflection. Connecting the demonstration tools to real services, adding error handling, and expanding the escalation policies would make it more suitable for production use.
